In [ ]:
!python -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd /content/PaddleOCR
!pip install -r requirements.txt


In [ ]:
# آماده سازی دیتاست
import random
import shutil
import unicodedata
from pathlib import Path

import yaml
from datasets import Image as HFImage
from datasets import get_dataset_split_names, load_dataset
from tqdm.auto import tqdm

DATASET = "AliShafiee2003/persian-ocr-garshasp-70c"
TOTAL = 10_000
VAL_TOTAL = 500
MAX_TEXT_LENGTH = 72
CTC_STEPS = 80
IMAGE_HEIGHT = 48
IMAGE_WIDTH = 640
BATCH_SIZE = 32
EPOCHS = 15
TEST_SAMPLES = 10
SEED = 42

ROOT = Path("/content/garshasp")
CONFIG = Path("/content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/garshasp.yaml")
OUTPUT = Path("/content/drive/MyDrive/PaddleOCR/Garshasp/arabic_PP-OCRv5_mobile_rec")
PRETRAINED = Path("/content/pretrained/arabic_PP-OCRv5_mobile_rec_pretrained.pdparams")
ARABIC_DICT = Path("/content/PaddleOCR/ppocr/utils/dict/ppocrv5_arabic_dict.txt")
PERSIAN_DICT = OUTPUT / "ppocrv5_persian_dict.txt"
BEST = OUTPUT / "best_accuracy"
EXPORT_DIR = OUTPUT / "inference"

ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
PRETRAINED.parent.mkdir(parents=True, exist_ok=True)
PERSIAN_DICT.write_bytes(ARABIC_DICT.read_bytes())

styles = get_dataset_split_names(DATASET)
allowed = set(PERSIAN_DICT.read_text(encoding="utf-8").splitlines()) | {" "}
translation = str.maketrans("يىك", "ییک", "\u200b\u200c\u200d\u200e\u200f\u202a\u202b\u202c\u202d\u202e\u2060\u2066\u2067\u2068\u2069\ufeff")


def normalize(text):
    return " ".join(unicodedata.normalize("NFC", text).translate(translation).split())


def ctc_steps(text):
    return len(text) + sum(a == b for a, b in zip(text, text[1:]))


rows = []

for style_number, style in enumerate(styles):
    target = TOTAL // len(styles) + (style_number < TOTAL % len(styles))
    (ROOT / "images" / style).mkdir(parents=True, exist_ok=True)

    stream = load_dataset(DATASET, split=style, streaming=True)
    stream = stream.cast_column("image", HFImage(decode=False))
    stream = stream.shuffle(seed=SEED + style_number, buffer_size=10_000)
    accepted = 0

    with tqdm(total=target, desc=style) as progress:
        for example in stream:
            text = normalize(example["text"])

            if (
                not text
                or len(text) > MAX_TEXT_LENGTH
                or ctc_steps(text) > CTC_STEPS
                or any(char not in allowed for char in text)
            ):
                continue

            relative = Path("images", style, f"{accepted:05d}.jpg")
            (ROOT / relative).write_bytes(example["image"]["bytes"])
            rows.append((relative.as_posix(), text))
            accepted += 1
            progress.update()

            if accepted == target:
                break


print(len(rows))

random.Random(SEED).shuffle(rows)
validation = rows[:VAL_TOTAL]
training = rows[VAL_TOTAL:]

for filename, samples in (("train.txt", training), ("val.txt", validation)):
    (ROOT / filename).write_text(
        "".join(f"{image}\t{text}\n" for image, text in samples),
        encoding="utf-8",
    )

print(len(training), len(validation))


In [ ]:
# یکسان کردن تعداد گام‌های CTC در train و eval
backbone = Path("/content/PaddleOCR/ppocr/modeling/backbones/rec_lcnetv3.py")
source = backbone.read_text(encoding="utf-8").replace(
    "F.adaptive_avg_pool2d(x, [1, 40])",
    f"F.adaptive_avg_pool2d(x, [1, {CTC_STEPS}])",
)
backbone.write_text(source, encoding="utf-8")


In [ ]:
# بررسی patch و آزاد کردن حافظه GPU
import paddle
from ppocr.modeling.backbones.rec_lcnetv3 import PPLCNetV3

model = PPLCNetV3(scale=0.95)
image = paddle.randn([1, 3, IMAGE_HEIGHT, IMAGE_WIDTH])

model.train()
train_steps = model(image).shape[-1]
model.eval()
eval_steps = model(image).shape[-1]




print("train:", train_steps, "eval:", eval_steps)
assert train_steps == eval_steps == CTC_STEPS

del model, image
paddle.device.cuda.empty_cache()


In [ ]:
# ساخت تنظیمات آموزش
BASE_CONFIG = Path("/content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/arabic_PP-OCRv5_mobile_rec.yaml")
config = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))


config["Global"].update(
    epoch_num=EPOCHS,
    save_model_dir=str(OUTPUT),
    save_epoch_step=1,
    eval_batch_step=[0, len(training) // BATCH_SIZE],
    max_text_length=MAX_TEXT_LENGTH,
    d2s_train_image_shape=[3, IMAGE_HEIGHT, IMAGE_WIDTH],
    character_dict_path=str(PERSIAN_DICT),
    pretrained_model=str(PRETRAINED),
)
config["Metric"]["main_indicator"] = "norm_edit_dis"
config["Architecture"]["Head"]["head_list"][1]["NRTRHead"]["max_text_length"] = MAX_TEXT_LENGTH

train = config["Train"]
train["dataset"].update(data_dir=str(ROOT), label_file_list=[str(ROOT / "train.txt")])
train["dataset"]["transforms"][1]["RecConAug"]["prob"] = 0.0
train["sampler"].update(scales=[[IMAGE_WIDTH, height] for height in (32, 48, 64)], first_bs=BATCH_SIZE)

evaluation = config["Eval"]
evaluation["dataset"].update(data_dir=str(ROOT), label_file_list=[str(ROOT / "val.txt")])
evaluation["dataset"]["transforms"][2]["RecResizeImg"]["image_shape"] = [3, IMAGE_HEIGHT, IMAGE_WIDTH]
evaluation["loader"]["batch_size_per_card"] = BATCH_SIZE

CONFIG.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(CONFIG)


In [ ]:
# دانلود مدل pretrained رسمی
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/arabic_PP-OCRv5_mobile_rec_pretrained.pdparams -O "{PRETRAINED}"


In [ ]:
# ارزیابی قبل از آموزش
!python tools/eval.py -c "{CONFIG}"


In [ ]:
# آموزش
!python tools/train.py -c "{CONFIG}"


In [ ]:
# ارزیابی بهترین checkpoint
!python tools/eval.py -c "{CONFIG}" -o Global.checkpoints="{BEST}"


In [ ]:
# export بهترین مدل
!python tools/export_model.py -c "{CONFIG}" -o Global.pretrained_model="{BEST}" Global.save_inference_dir="{EXPORT_DIR}" Global.model_name="PP-OCRv5_mobile_rec"


In [ ]:
# آماده کردن نمونه‌های validation برای تست
TEST_DIR = Path("/content/export_test")
shutil.rmtree(TEST_DIR, ignore_errors=True)
TEST_DIR.mkdir()

for index, line in enumerate((ROOT / "val.txt").read_text(encoding="utf-8").splitlines()[:TEST_SAMPLES]):
    relative_path, truth = line.split("\t", 1)
    destination = TEST_DIR / f"{index:02d}{Path(relative_path).suffix}"
    shutil.copy2(ROOT / relative_path, destination)
    print(destination.name, "GT:", truth)

print("Model:", EXPORT_DIR)


In [ ]:
# تست مدل export شده
!python tools/infer/predict_rec.py --image_dir="{TEST_DIR}" --rec_model_dir="{EXPORT_DIR}" --rec_algorithm="SVTR_LCNet" --rec_image_shape="3,{IMAGE_HEIGHT},{IMAGE_WIDTH}" --rec_char_dict_path="{PERSIAN_DICT}" --use_gpu=True
